# FastAPI Integration

In this notebook, we prepare the trained CarSight model for integration with a FastAPI backend.

The goal is to define the request schema, reuse the prediction logic, and simulate how a `/predict` endpoint will process vehicle data and return a predicted price.

In [9]:
import joblib
import pandas as pd
from fastapi import FastAPI

from pydantic import BaseModel

In [2]:
model = joblib.load("../models/random_forest_model.pkl")

feature_columns = model.feature_names_in_

In [3]:
class CarRequest(BaseModel):
    marka: str
    yıl: int
    kilometre_Km: int
    vitesTipi: str
    yakitTuru: str
    kasaTipi: str

In [4]:
def prepare_input(car: CarRequest):

    car_data = {
        "marka": car.marka,
        "yıl": car.yıl,
        "kilometre(Km)": car.kilometre_Km,
        "vitesTipi": car.vitesTipi,
        "yakitTuru": car.yakitTuru,
        "kasaTipi": car.kasaTipi,
    }

    input_df = pd.DataFrame([car_data])

    input_encoded = pd.get_dummies(input_df)

    input_encoded = input_encoded.reindex(
        columns=feature_columns,
        fill_value=False
    )

    return input_encoded

In [5]:
def predict_price(car: CarRequest):

    input_encoded = prepare_input(car)

    prediction = model.predict(input_encoded)

    return int(prediction[0])

In [6]:
sample_request = CarRequest(
    marka="BMW",
    yıl=2020,
    kilometre_Km=80000,
    vitesTipi="Otomatik",
    yakitTuru="Dizel",
    kasaTipi="Sedan"
)

In [7]:
predicted_price = predict_price(sample_request)

print(f"Predicted Price: {predicted_price:,} TL")

Predicted Price: 2,483,634 TL


In [8]:
response = {
    "predicted_price": predicted_price
}

response

{'predicted_price': 2483634}

In [10]:
app = FastAPI(
    title="CarSight API",
    version="1.0.0"
)

In [11]:
@app.post("/predict")
def predict(car: CarRequest):

    predicted_price = predict_price(car)

    return {
        "predicted_price": predicted_price
    }

In [12]:
@app.get("/health")
def health():
    return {
        "status": "ok",
        "service": "carsight-api"
    }

# Conclusion

The model prediction logic is now wrapped in a FastAPI-compatible structure.

The next step is to move this implementation into the CarSight backend and expose the `/predict` endpoint as a real API route.